<div style="display:flex; align-items:center; gap:10px; margin-bottom:8px;">
  <span style="font-size:26px; color:#9558B2;">●</span>
  <span style="font-size:26px; color:#389826;">●</span>
  <span style="font-size:26px; color:#CB3C33;">●</span>
  <span style="font-size:26px; color:#4063D8;">●</span>
  <span style="font-size:30px; font-weight:700; margin-left:6px;">Julia</span>
</div>

# Julia od zera — **Lesson 8**
## 📘 **Multiple Dispatch** — metody, typy, selekcja metod i projektowanie API

**Cartesian School · Julia Course**  
**Autor:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School


## Informacje o lekcji

| Pole | Wartość |
|---|---|
| Kurs | Julia od zera |
| Numer lekcji | Lesson 8 |
| Tytuł | Multiple Dispatch |
| Poziom | Początkujący+ / średniozaawansowany |
| Szacowany czas | 180–240 minut |
| Wymagania | Lesson 0–7 |
| Zakres | funkcje i metody, typy abstrakcyjne i konkretne, single vs multiple dispatch, selekcja najbardziej specyficznej metody, `methods`, `@which`, fallback, ambiguities, parametric methods, `where`, traits, promotion, projektowanie API |
| Autor | Siergej Sobolewski |
| Prawa | © 2026 Cartesian School |

---

## Plan lekcji

1. Dlaczego multiple dispatch jest ważny.
2. Funkcja a metoda.
3. Single dispatch i multiple dispatch.
4. Pierwsze przeciążone metody.
5. Typy abstrakcyjne i konkretne.
6. Najbardziej specyficzna metoda.
7. Fallback methods.
8. `methods`.
9. `@which`.
10. `applicable`.
11. `hasmethod`.
12. Dispatch dla wielu argumentów.
13. Przykład geometryczny.
14. Parametryczne metody i `where`.
15. Ograniczanie typów.
16. Dispatch na wartościach przez typy.
17. `Val`.
18. Ambiguities.
19. Jak unikać niejednoznaczności.
20. Promotion i dispatch.
21. `convert` i dispatch.
22. Traits.
23. Dispatch a `if x isa`.
24. Dispatch a wydajność.
25. Projektowanie rozszerzalnego API.
26. Typowe błędy.
27. Praktyka.
28. Mini-projekt.
29. Checkpoint.
30. Podsumowanie.


## Standard dydaktyczny Cartesian School

W całym kursie stosujemy spójny układ:

| Oznaczenie | Znaczenie |
|---|---|
| **Cel** | czego nauczysz się w danym fragmencie |
| **Teoria** | definicje i reguły |
| **Przykład** | minimalny, działający kod |
| **Analiza** | wyjaśnienie działania |
| **Ważne** | zasada, którą trzeba zapamiętać |
| **Typowy błąd** | częsty błąd i jego przyczyna |
| **Spróbuj sam** | mały eksperyment |
| **Praktyka** | zadanie do samodzielnego wykonania |
| **Podsumowanie** | najważniejsze wnioski |


## Cele lekcji

Po ukończeniu Lesson 8 będziesz potrafić:

- wyjaśnić różnicę między funkcją a metodą;
- rozumieć, dlaczego multiple dispatch jest centralnym mechanizmem Julia;
- definiować wiele metod tej samej funkcji;
- przewidywać, która metoda zostanie wybrana;
- korzystać z `methods`, `@which`, `applicable` i `hasmethod`;
- projektować fallback methods;
- pisać parametryczne metody z `where`;
- rozpoznawać i usuwać ambiguities;
- rozumieć rolę promotion i conversion;
- porównać dispatch z dużym `if x isa`;
- projektować API, które można rozszerzać bez modyfikowania istniejącego kodu.


## **1. Dlaczego multiple dispatch jest ważny?**

### Teoria

Julia jest językiem, w którym funkcje nie są „własnością” pojedynczego typu.

Zachowanie programu możemy definiować poprzez **metody**, których wybór zależy od typów wszystkich argumentów funkcji.

To właśnie nazywamy **multiple dispatch**.


### Intuicja

Załóżmy, że mamy funkcję:

```julia
combine(x, y)
```

Jej zachowanie może zależeć jednocześnie od typu `x` i typu `y`.

Dzięki temu nie musimy budować jednego wielkiego bloku:

```julia
if x isa ...
    if y isa ...
        ...
    end
end
```

Zamiast tego definiujemy osobne metody.


## **2. Funkcja a metoda**

### Teoria

To podstawowe rozróżnienie:

- **funkcja** — wspólna nazwa operacji;
- **metoda** — konkretna implementacja dla określonego zestawu typów argumentów.


In [ ]:
describe(x::Integer) = "liczba całkowita"
describe(x::AbstractFloat) = "liczba zmiennoprzecinkowa"
describe(x::AbstractString) = "napis"

@show describe(10)
@show describe(3.14)
@show describe("Julia")


W tym przykładzie istnieje jedna funkcja `describe`, ale kilka metod.


## **3. Single dispatch i multiple dispatch**

### Single dispatch

W wielu językach obiektowych metoda jest wybierana głównie na podstawie typu jednego obiektu — zwykle tego po lewej stronie kropki.


### Multiple dispatch

W Julia wybór może zależeć od **wszystkich argumentów**.


In [ ]:
interact(x::Integer, y::Integer) = "Integer + Integer"
interact(x::Integer, y::AbstractString) = "Integer + String"
interact(x::AbstractString, y::Integer) = "String + Integer"
interact(x::AbstractString, y::AbstractString) = "String + String"

@show interact(1, 2)
@show interact(1, "Julia")
@show interact("Julia", 1)
@show interact("Julia", "Lang")


## **4. Pierwsze przeciążone metody**

### Przykład


In [ ]:
combine(x::Integer, y::Integer) = x + y
combine(x::AbstractString, y::AbstractString) = x * y

@show combine(10, 20)
@show combine("Julia ", "Language")


### Analiza

Ta sama nazwa `combine` reprezentuje różne znaczenia w zależności od typów argumentów.


## **5. Typy abstrakcyjne i konkretne**

### Teoria

Julia posiada hierarchię typów.

Przykładowo:

```text
Number
├── Real
│   ├── Integer
│   └── AbstractFloat
└── Complex
```

Typy abstrakcyjne opisują **rodzinę typów**, a typy konkretne reprezentują faktyczne typy wartości.


In [ ]:
@show Int <: Integer
@show Integer <: Real
@show Float64 <: AbstractFloat
@show Float64 <: Real


### Ważne

Metoda przyjmująca `Real` może działać dla wielu typów liczbowych, np. `Int`, `Float64`, `BigFloat`.

Nie trzeba definiować osobnej metody dla każdego typu konkretnego, jeśli zachowanie jest takie samo.


## **6. Najbardziej specyficzna metoda**

### Teoria

Jeżeli pasuje kilka metod, Julia wybiera metodę **najbardziej specyficzną**.


In [ ]:
kind(x::Number) = "Number"
kind(x::Real) = "Real"
kind(x::Integer) = "Integer"
kind(x::Int) = "Int"

@show kind(1)
@show kind(Int8(1))
@show kind(1.0)
@show kind(1 + 2im)


### Analiza

Dla `1::Int` pasują wszystkie cztery metody, ale `Int` jest najbardziej specyficzne.


## **7. Fallback methods**

### Teoria

Fallback method to metoda ogólna używana wtedy, gdy nie istnieje bardziej specyficzna wersja.


In [ ]:
describe_type(x::Integer) = "integer"
describe_type(x::AbstractString) = "string"
describe_type(x) = "other"

@show describe_type(10)
@show describe_type("Julia")
@show describe_type([1, 2, 3])


### Dobra praktyka

Fallback powinien mieć sens semantyczny.

Nie dodawaj `f(x) = ...` tylko po to, aby ukryć błąd projektu typów.


## **8. Inspekcja metod — `methods`**

### Przykład


In [ ]:
methods(describe_type)


### Zastosowanie

`methods(f)` pokazuje wszystkie znane metody funkcji `f`.

To bardzo ważne narzędzie do nauki i debugowania dispatch.


## **9. `@which` — która metoda zostanie wywołana?**

### Przykład


In [ ]:
@which describe_type(10)


In [ ]:
@which describe_type("Julia")


### Analiza

`@which` pozwala bezpośrednio zobaczyć metodę wybraną dla konkretnych argumentów.


## **10. `applicable`**

### Teoria

`applicable(f, args...)` sprawdza, czy istnieje metoda możliwa do wywołania dla danych argumentów.


In [ ]:
only_ints(x::Integer, y::Integer) = x + y

@show applicable(only_ints, 1, 2)
@show applicable(only_ints, 1.0, 2.0)


## **11. `hasmethod`**

### Teoria

`hasmethod` pozwala sprawdzić, czy funkcja ma metodę zgodną z określonym tuple typów.


In [ ]:
@show hasmethod(only_ints, Tuple{Int, Int})
@show hasmethod(only_ints, Tuple{Float64, Float64})


## **12. Dispatch dla wielu argumentów**

### Przykład


In [ ]:
compare_values(x::Integer, y::Integer) = "dwie liczby całkowite"
compare_values(x::Real, y::Real) = "dwie liczby rzeczywiste"
compare_values(x::Real, y::AbstractString) = "liczba i napis"
compare_values(x, y) = "inne połączenie"

@show compare_values(1, 2)
@show compare_values(1.5, 2.5)
@show compare_values(1, "Julia")
@show compare_values([1], :symbol)


### Ważne

Wybór nie zależy od „pierwszego argumentu plus reszta”.

Julia analizuje cały zestaw typów argumentów.


## **13. Przykład geometryczny**

### Typy


In [ ]:
abstract type Shape end

struct Circle <: Shape
    radius::Float64
end

struct Rectangle <: Shape
    width::Float64
    height::Float64
end


### Pole powierzchni


In [ ]:
area(s::Circle) = π * s.radius^2
area(s::Rectangle) = s.width * s.height

c = Circle(2.0)
r = Rectangle(3.0, 4.0)

@show area(c)
@show area(r)


### Interakcja dwóch figur


In [ ]:
relation(a::Circle, b::Circle) = "dwa okręgi"
relation(a::Rectangle, b::Rectangle) = "dwa prostokąty"
relation(a::Circle, b::Rectangle) = "okrąg i prostokąt"
relation(a::Rectangle, b::Circle) = "prostokąt i okrąg"

@show relation(c, Circle(1.0))
@show relation(c, r)
@show relation(r, c)


## **14. Parametryczne metody i `where`**

### Przykład


In [ ]:
same_type_add(x::T, y::T) where {T<:Number} = x + y

@show same_type_add(1, 2)
@show same_type_add(1.5, 2.5)


### Analiza

`where {T<:Number}` oznacza:

- istnieje pewien typ `T`;
- `T` jest podtypem `Number`;
- oba argumenty mają dokładnie ten sam typ `T`.


## **15. Ograniczanie typów — kiedy ma sens?**

### Dobra praktyka

Ograniczenia typów dodajemy wtedy, gdy:

- zachowanie rzeczywiście zależy od typu;
- chcemy zdefiniować konkretną metodę;
- potrzebujemy jednoznacznego kontraktu.


### Typowy błąd

Nie pisz:

```julia
f(x::Int) = x + 1
```

jeżeli funkcja równie dobrze działa dla wszystkich liczb.

Lepsza wersja może być:


In [ ]:
increment_number(x::Number) = x + one(x)

@show increment_number(10)
@show increment_number(3.5)


## **16. Dispatch zależny od wartości — przez typy**

### Teoria

Dispatch w Julia działa na **typach**, nie bezpośrednio na arbitralnych wartościach.

Jeżeli zachowanie ma zależeć od wartości, zwykle stosujemy:

- zwykły `if`;
- albo kodujemy wartość w typie.


### Przykład z typami znacznikowymi


In [ ]:
abstract type Operation end
struct AddOperation <: Operation end
struct MultiplyOperation <: Operation end

apply(::AddOperation, x, y) = x + y
apply(::MultiplyOperation, x, y) = x * y

@show apply(AddOperation(), 3, 4)
@show apply(MultiplyOperation(), 3, 4)


## **17. `Val` — wartość jako część typu**

### Teoria

`Val{x}` pozwala przenieść małą wartość do systemu typów.


In [ ]:
operation(::Val{:add}, x, y) = x + y
operation(::Val{:mul}, x, y) = x * y

@show operation(Val(:add), 2, 3)
@show operation(Val(:mul), 2, 3)


### Ważne

`Val` jest narzędziem specjalistycznym.

Nie należy zastępować nim zwykłych warunków tylko dlatego, że istnieje.


## **18. Niejednoznaczność metod — ambiguities**

### Teoria

Ambiguity powstaje, gdy dla danego wywołania dwie metody są równie specyficzne i Julia nie może jednoznacznie wybrać jednej.


### Przykład problemu


In [ ]:
ambiguous_demo(x::Integer, y::Real) = "Integer, Real"
ambiguous_demo(x::Real, y::Integer) = "Real, Integer"


Wywołanie:

```julia
ambiguous_demo(1, 1)
```

pasuje do obu metod.

Aby uniknąć pozostawiania notebooka w stanie błędu, sprawdzimy to bezpiecznie:


In [ ]:
ambiguity_detected = try
    ambiguous_demo(1, 1)
    false
catch e
    e isa MethodError
end

@show ambiguity_detected


## **19. Jak usuwać ambiguities?**

### Rozwiązanie

Dodaj najbardziej specyficzną metodę:


In [ ]:
ambiguous_demo(x::Integer, y::Integer) = "Integer, Integer"

@show ambiguous_demo(1, 1)


### Dobra praktyka

Jeżeli dwa przecięcia typów są logicznie możliwe, zdefiniuj metodę dla ich przecięcia.


## **20. Promotion i dispatch**

### Teoria

W obliczeniach numerycznych często mamy różne typy:

```julia
1       # Int
2.5     # Float64
```

Julia posiada mechanizm promocji typów, który pozwala znaleźć wspólny typ reprezentacji.


In [ ]:
@show promote(1, 2.5)
@show promote_type(Int, Float64)


### Przykład z własną funkcją


In [ ]:
function add_promoted(x::Number, y::Number)
    xp, yp = promote(x, y)
    return xp + yp
end

@show add_promoted(1, 2.5)


## **21. `convert` i dispatch**

### Teoria

`convert(T, x)` sam jest funkcją korzystającą z metod.

Możemy rozszerzyć mechanizm konwersji dla własnego typu.


In [ ]:
struct Celsius
    value::Float64
end

Base.convert(::Type{Celsius}, x::Real) = Celsius(Float64(x))

celsius = convert(Celsius, 36)

@show celsius
@show celsius.value


### Ważne

Rozszerzając funkcje z `Base`, trzeba zachować poprawną semantykę i istniejące konwencje.


## **22. Traits — wzorzec projektowy oparty na dispatch**

### Teoria

Trait to sposób opisania właściwości typu, która niekoniecznie wynika bezpośrednio z hierarchii dziedziczenia.


In [ ]:
abstract type StorageTrait end
struct MutableStorage <: StorageTrait end
struct ImmutableStorage <: StorageTrait end

storage_trait(::Type{<:AbstractVector}) = MutableStorage()
storage_trait(::Type{<:Tuple}) = ImmutableStorage()

storage_message(x) = storage_message(storage_trait(typeof(x)), x)

storage_message(::MutableStorage, x) = "kolekcja mutowalna"
storage_message(::ImmutableStorage, x) = "kolekcja niemutowalna"

@show storage_message([1, 2, 3])
@show storage_message((1, 2, 3))


### Analiza

Najpierw określamy trait, a następnie dispatchujemy na jego typie.

To zaawansowany, ale bardzo użyteczny wzorzec w projektowaniu rozszerzalnych bibliotek.


## **23. Dispatch a `if x isa ...`**

### Podejście warunkowe


In [ ]:
function describe_with_if(x)
    if x isa Integer
        return "integer"
    elseif x isa AbstractFloat
        return "float"
    elseif x isa AbstractString
        return "string"
    else
        return "other"
    end
end


### Podejście przez dispatch


In [ ]:
describe_dispatch(x::Integer) = "integer"
describe_dispatch(x::AbstractFloat) = "float"
describe_dispatch(x::AbstractString) = "string"
describe_dispatch(x) = "other"

@assert describe_with_if(10) == describe_dispatch(10)
@assert describe_with_if(3.14) == describe_dispatch(3.14)
@assert describe_with_if("Julia") == describe_dispatch("Julia")


### Kiedy wybrać które podejście?

- decyzja zależy od **wartości** → `if`;
- decyzja zależy od **typu** → często dispatch;
- prosty jednorazowy warunek → `if` może być czytelniejszy;
- rozszerzalne API → dispatch zwykle skaluje się lepiej.


## **24. Dispatch a wydajność**

### Teoria

Multiple dispatch nie jest tylko mechanizmem organizacji kodu.

W połączeniu z kompilacją specjalizowaną pozwala Julii generować kod dopasowany do konkretnych typów argumentów.


### Ważne

Nie oznacza to, że „im więcej metod, tym szybciej”.

Wydajność zależy m.in. od:

- stabilności typów;
- przewidywalności dispatch;
- alokacji;
- struktury danych;
- sposobu napisania kodu.


## **25. Projektowanie rozszerzalnego API**

### Cel

Dobrze zaprojektowana funkcja powinna umożliwiać dodawanie nowych typów bez modyfikowania istniejącego kodu.


In [ ]:
abstract type Animal end

struct Dog <: Animal
    name::String
end

struct Cat <: Animal
    name::String
end

speak(a::Dog) = "$(a.name): hau!"
speak(a::Cat) = "$(a.name): miau!"

@show speak(Dog("Rex"))
@show speak(Cat("Luna"))


### Rozszerzenie przez nowy typ

Dodajemy nowy typ i metodę:


In [ ]:
struct Duck <: Animal
    name::String
end

speak(a::Duck) = "$(a.name): kwa!"

@show speak(Duck("Donald"))


### Analiza

Nie musieliśmy zmieniać istniejącej funkcji warunkowej.

Dodaliśmy nową metodę dla nowego typu.

To jedna z największych zalet multiple dispatch.


## **26. Typowe błędy początkujących**

| Błąd | Przyczyna | Poprawne podejście |
|---|---|---|
| mylenie funkcji z metodą | jedna funkcja może mieć wiele implementacji | użyj `methods(f)` |
| zbyt wąskie typy | funkcja niepotrzebnie ograniczona | użyj typu abstrakcyjnego |
| ogromny `if x isa ...` | zachowanie zależy od typów | rozważ dispatch |
| brak fallback | nieobsłużony typ kończy się `MethodError` | dodaj fallback, jeśli ma sens |
| zbyt szeroki fallback | ukrywa błędy projektu | stosuj świadomie |
| ambiguities | przecinające się sygnatury | dodaj bardziej specyficzną metodę |
| używanie `Val` do wszystkiego | komplikacja projektu | stosuj tylko tam, gdzie to uzasadnione |
| oczekiwanie dispatch po wartości | dispatch działa po typach | użyj `if` lub encode value in type |


## **27. Praktyka**

### Zadanie 27.1 — `describe_number`

Zdefiniuj funkcję z metodami dla:

- `Integer`,
- `AbstractFloat`,
- `Complex`,
- fallback dla innych typów.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 27.1


In [ ]:
describe_number(x::Integer) = "integer"
describe_number(x::AbstractFloat) = "float"
describe_number(x::Complex) = "complex"
describe_number(x) = "other"

@assert describe_number(1) == "integer"
@assert describe_number(1.5) == "float"
@assert describe_number(1 + 2im) == "complex"
@assert describe_number("Julia") == "other"


### Zadanie 27.2 — dwa argumenty

Zdefiniuj `combine_values(x, y)` tak, aby:

- dwa `Integer` były dodawane;
- dwa `String` były konkatenowane;
- pozostałe kombinacje zwracały `"unsupported"`.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 27.2


In [ ]:
combine_values(x::Integer, y::Integer) = x + y
combine_values(x::AbstractString, y::AbstractString) = x * y
combine_values(x, y) = "unsupported"

@assert combine_values(2, 3) == 5
@assert combine_values("Julia ", "Lang") == "Julia Lang"
@assert combine_values(1, "Julia") == "unsupported"


### Zadanie 27.3 — `@which`

Sprawdź, która metoda `combine_values` zostanie użyta dla:

```julia
combine_values(2, 3)
```


In [ ]:
@which combine_values(2, 3)


### Zadanie 27.4 — ambiguity

Utwórz dwie przecinające się metody:

```julia
f(x::Integer, y::Real)
f(x::Real, y::Integer)
```

Następnie dodaj trzecią metodę rozwiązującą konflikt dla `(Integer, Integer)`.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 27.4


In [ ]:
dispatch_test(x::Integer, y::Real) = "Integer, Real"
dispatch_test(x::Real, y::Integer) = "Real, Integer"
dispatch_test(x::Integer, y::Integer) = "Integer, Integer"

@assert dispatch_test(1, 1) == "Integer, Integer"


### Zadanie 27.5 — geometria

Dodaj typ:

```julia
struct Square <: Shape
    side::Float64
end
```

oraz metodę `area`.


In [ ]:
# Twoje rozwiązanie:


### Przykładowe rozwiązanie 27.5


In [ ]:
struct Square <: Shape
    side::Float64
end

area(s::Square) = s.side^2

@assert area(Square(5.0)) == 25.0


## **28. Mini-projekt — system płatności oparty na dispatch**

### Cel

Zaprojektujemy API, w którym różne metody płatności mają różne zachowanie.


In [ ]:
abstract type PaymentMethod end

struct CardPayment <: PaymentMethod
    last4::String
end

struct BankTransfer <: PaymentMethod
    iban::String
end

struct CashPayment <: PaymentMethod
end


### Metody `pay`


In [ ]:
pay(method::CardPayment, amount::Real) =
    "Płatność kartą ****$(method.last4): $(round(amount, digits=2)) PLN"

pay(method::BankTransfer, amount::Real) =
    "Przelew na $(method.iban): $(round(amount, digits=2)) PLN"

pay(::CashPayment, amount::Real) =
    "Płatność gotówką: $(round(amount, digits=2)) PLN"


In [ ]:
card = CardPayment("4242")
transfer = BankTransfer("PL001234567890")
cash = CashPayment()

@show pay(card, 199.99)
@show pay(transfer, 500)
@show pay(cash, 50)


### Rozszerzenie systemu

Dodajemy nową metodę płatności bez zmiany istniejących metod:


In [ ]:
struct CryptoPayment <: PaymentMethod
    asset::Symbol
end

pay(method::CryptoPayment, amount::Real) =
    "Płatność $(method.asset): $(round(amount, digits=2)) PLN"

@show pay(CryptoPayment(:BTC), 1000)


### Analiza

Mini-projekt pokazuje kluczową własność multiple dispatch:

**system można rozszerzać przez dodawanie nowych typów i metod bez modyfikowania starego kodu.**


## **29. Checkpoint końcowy**

Odpowiedz bez uruchamiania kodu:

1. Czym różni się funkcja od metody?
2. Co oznacza multiple dispatch?
3. Na ilu argumentach może opierać się wybór metody?
4. Co oznacza „najbardziej specyficzna metoda”?
5. Czym różni się typ abstrakcyjny od konkretnego?
6. Po co stosuje się fallback methods?
7. Co pokazuje `methods(f)`?
8. Do czego służy `@which`?
9. Co sprawdza `applicable`?
10. Co sprawdza `hasmethod`?
11. Co oznacza `where {T<:Number}`?
12. Czy dispatch działa bezpośrednio na wartościach?
13. Do czego służy `Val`?
14. Co to jest ambiguity?
15. Jak usunąć ambiguity?
16. Do czego służy promotion?
17. Jakie relacje ma `convert` z dispatch?
18. Czym jest trait?
19. Kiedy lepszy jest `if`, a kiedy dispatch?
20. Dlaczego multiple dispatch dobrze wspiera rozszerzalne API?


## **30. Podsumowanie lekcji**

Najważniejsze zasady Lesson 8:

1. Funkcja może mieć wiele metod.
2. Julia wybiera metodę na podstawie typów wszystkich argumentów.
3. Najbardziej specyficzna pasująca metoda ma pierwszeństwo.
4. Typy abstrakcyjne umożliwiają definiowanie metod dla całych rodzin typów.
5. Fallback methods pomagają obsługiwać przypadki ogólne.
6. `methods`, `@which`, `applicable` i `hasmethod` są kluczowymi narzędziami inspekcji.
7. `where` pozwala definiować zależności między typami argumentów.
8. Dispatch działa po typach, nie po arbitralnych wartościach.
9. `Val` może przenieść małą wartość do systemu typów, ale nie należy go nadużywać.
10. Ambiguities powstają przy przecinających się, równie specyficznych metodach.
11. Promotion pomaga ujednolicić typy w operacjach numerycznych.
12. `convert` również opiera się na mechanizmie metod.
13. Traits pozwalają modelować właściwości niezależne od prostej hierarchii typów.
14. Jeżeli zachowanie zależy od typu, dispatch często jest lepszy niż duży `if x isa`.
15. Multiple dispatch jest jednym z najważniejszych mechanizmów projektowania rozszerzalnych bibliotek Julia.


## Źródła do dalszego czytania

- Julia Manual — Methods
- Julia Manual — Types
- Julia Manual — Conversion and Promotion
- Julia Manual — Interfaces
- Julia Base — `methods`
- Julia InteractiveUtils — `@which`
- Julia Base — `applicable`
- Julia Base — `hasmethod`


---

**Cartesian School · Julia Course**  
**Lesson 8 — Multiple Dispatch**  
**Autor:** Siergej Sobolewski  
**Copyright:** © 2026 Cartesian School

[← Lesson 7 — Plotting](Lesson_7_Plotting_Julia_Cartesian_School_Professional_PL.ipynb)  
[Spis treści](README.md)  
[Lesson 9 — Basic Linear Algebra →](10%20-%20Basic%20linear%20algebra.ipynb)
